# Sistem Pakar Rekomendasi BUY vs SELL Saham Indonesia (IHSG & LQ45)
## Berbasis Perbandingan Sentimen Headline vs Isi Berita (Bilingual FiGAS: Media Nasional & Reuters) + Certainty Factor MYCIN

**Tim Pengembang (Proyek Sistem Pakar - Informatika UKRIDA):**
- 412024002 - Owen Antony
- 412024005 - Kristofer Ryan Giggs Eka Saputra
- 412024008 - Steven Iwan
- 412024032 - Gaddiel Abiyr Nesher Sitorus

---
### Ringkasan Perbaikan berdasarkan Evaluasi & Masukan Dosen:
1. **Fokus Pasar Indonesia & Data 2 Tahun Terakhir (2024–2026)**: Menganalisis **IHSG (`^JKSE`)** dan **6 Saham Unggulan LQ45** (`BBCA.JK`, `BBRI.JK`, `BMRI.JK`, `TLKM.JK`, `ASII.JK`, `ADRO.JK`) menggunakan data harga harian `yfinance` serta berita ekonomi nasional (CNBC Indonesia, Antara, Detik Finance, Kontan) dan internasional (**Reuters**).
2. **Inovasi Perbandingan Headline vs Isi Berita (*Clickbait Discount & Content Verification*)**: Mengatasi bias judul berita yang sering hiperbolis (*clickbait*) dengan membandingkan skor sentimen judul ($S_{\text{head}}$) dan isi artikel ($S_{\text{body}}$).
3. **Keputusan Biner Tegas (`BUY` vs `SELL`)**: Menyederhanakan keputusan menjadi `BUY` dan `SELL` sesuai arahan dosen.
4. **14 Aturan Pakar Ber-Referensi Literatur ([1]–[8]) & Penjelasan Alasan (*Reason to Buy / Reason to Sell*)**: Seluruh aturan merujuk pada literatur ilmiah dan mengeluarkan penjelasan bahasa manusia.
5. **Evaluasi Terhadap Harga Saham Asli Hari Berikutnya ($t+1$)**: Menguji akurasi keputusan hari ke-$t$ dengan pergerakan harga penutupan asli di $t+1$ (akurasi meningkat dari **41,5%** pada *Headline Saja* menjadi **62,1%** dengan *Verifikasi Headline + Isi Berita*).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from expert_system import (
    headline_sentiment,
    analyze_article_headline_vs_body,
    make_facts,
    fuzzify,
    RULES,
    ACADEMIC_REFERENCES,
    infer,
    run_test_cases,
    run_unit_tests,
    fetch_stock_indicators,
    fetch_live_news_for_ticker,
    load_2y_dataset,
)
print(f"Jumlah Aturan Pakar: {len(RULES)} Aturan (Murni BUY vs SELL)")
print(f"Jumlah Referensi Akademik Terkutip: {len(ACADEMIC_REFERENCES)} Referensi ([1] s/d [8])")

## 1. Demonstrasi Perbandingan Headline vs Isi Berita (Clickbait Discount)

In [2]:
contoh_berita = analyze_article_headline_vs_body(
    headline="IHSG Kebakaran dan Ambruk Rontok di Awal Sesi!",
    body="Walaupun judul berita terdengar panik, koreksi sebenarnya hanya tipis 0,12% akibat aksi ambil untung wajar, sedangkan laba perbankan tumbuh solid dan investor asing mencatat net buy.",
    source="Detik Finance",
    scope="MACRO_IHSG"
)
print("Status Verifikasi :", contoh_berita["verification_label"])
print("Skor Headline     :", contoh_berita["headline_score"])
print("Skor Isi Berita   :", contoh_berita["body_score"])
print("Skor Efektif Akhir:", contoh_berita["effective_score"])
print("Penjelasan        :", contoh_berita["verification_explanation"])

## 2. Fungsi Demonstrasi `advise_saham_indonesia(ticker)` (Live Request Saham & Berita)

In [3]:
def advise_saham_indonesia(ticker="BBCA.JK", max_articles=10):
    stock = fetch_stock_indicators(ticker)
    articles = fetch_live_news_for_ticker(ticker, max_articles=max_articles)
    facts = make_facts(articles, trend=stock["trend"], vol=stock["volatility"], rsi=stock["rsi14"], ticker=ticker)
    lf = fuzzify(facts)
    dec, cf, fired, trace = infer(lf, facts)
    print(f"=== REKOMENDASI SISTEM PAKAR: {stock['ticker']} ({stock['name']}) ===")
    print(f"Harga Terakhir : {stock['last_close']} | Tren vs MA-20: {stock['trend_pct']:+.2f}% | RSI-14: {stock['rsi14']}")
    print(f"Jumlah Berita  : {facts['n']} berita (Skor Headline: {facts['sent_headline_only']:+.3f} -> Skor Efektif Isi: {facts['sent_effective']:+.3f})")
    print(f"Keputusan Akhir: {dec} (CF BUY: {cf['BUY']:.2f} vs CF SELL: {cf['SELL']:.2f})")
    print("Alasan Rekomendasi (Reasoning):")
    for r in trace["primary_reasons"]:
        print("  ->", r)
    return dec, cf, trace

_ = advise_saham_indonesia("BBCA.JK", max_articles=8)

## 3. Evaluasi Historis 2 Tahun (2024–2026) terhadap Harga Saham Asli ($t+1$)

In [4]:
data_2y = load_2y_dataset()
m = data_2y["overall_metrics"]
print("Periode Evaluasi :", data_2y["period"])
print("Total Sampel Hari:", data_2y["total_samples"], "hari bursa | Total Artikel:", data_2y["total_articles_analyzed"])
print(f"Akurasi (Hanya Headline)         : {m['accuracy_headline_only']*100:.1f}% (Macro-F1: {m['macro_f1_headline_only']:.3f})")
print(f"Akurasi (Headline + Isi Berita)  : {m['accuracy_full']*100:.1f}% (Macro-F1: {m['macro_f1_full']:.3f})")
print(f"Presisi BUY: {m['precision_buy']*100:.1f}% | Presisi SELL: {m['precision_sell']*100:.1f}%")